## Qwen3-VL-8B Fold 0 LoRA 학습

이 노트북은 팀 공통 `fold_assignments.csv`를 그대로 읽어 Fold 0만 Validation으로 고정합니다. Fold를 다시 계산하지 않으며 Test 데이터에는 접근하지 않습니다.

첫 비교 실험은 Sinyoung 모델과 같은 Qwen3-VL-8B 백본·조건부 프롬프트·LoRA 기본값을 사용하되, 학습 데이터만 `원본 + 결정적 shuffled copy 1개`로 구성합니다. 한 epoch의 총 예시는 15,240개이므로 원본만 2 epochs 학습하는 것과 노출 수가 같습니다.

실행 순서는 환경 확인 → 공통 Fold 감사 → 증강/정답 변환 감사 → 모델/LoRA 감사 → 학습 → 전체 Fold 0 Exact Match 평가입니다.

### 1. A100, Google Drive, 공통 Fold 파일 확인

학습 전에 GPU와 필수 경로만 검사합니다. 이 셀은 데이터를 학습하거나 모델을 다운로드하지 않습니다. `fold_assignments.csv`는 앞 단계에서 Colab의 scikit-learn 1.6.1로 생성한 파일이어야 합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
assert torch.cuda.is_bf16_supported(), 'BF16을 지원하는 GPU가 필요합니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TRAIN_CSV = DATA_DIR / 'train.csv'
TRAIN_IMAGE_DIR = DATA_DIR / 'train'
FOLD_CSV = PROJECT_DIR / 'outputs' / 'splits' / 'fold_assignments.csv'

MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
RUN_NAME = 'fold0_aug1'
RUN_DIR = PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora' / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
FINAL_ADAPTER_DIR = RUN_DIR / 'final_adapter'

for required_path in (TRAIN_CSV, TRAIN_IMAGE_DIR, FOLD_CSV):
    assert required_path.exists(), f'필수 경로가 없습니다: {required_path}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'프로젝트: {PROJECT_DIR}')
print(f'공통 Fold: {FOLD_CSV}')
print(f'실험 출력: {RUN_DIR}')

### 2. Qwen3-VL LoRA 패키지 설치

Colab의 PyTorch와 CUDA는 교체하지 않고 Transformers, PEFT, bitsandbytes, Qwen 이미지 유틸리티만 설치합니다. 버전을 고정해 다음 실행에서도 같은 API를 사용합니다. 설치 후 import 문제가 발생할 때만 런타임을 재시작하고 1번 셀부터 다시 실행합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 bitsandbytes qwen-vl-utils==0.0.14 ipywidgets
%pip uninstall -y torchao

### 3. 공통 Fold 0과 Train 데이터 무결성 감사

저장된 Fold를 다시 계산하지 않고 `Id`로 Train과 병합합니다. 9,535개 ID가 정확히 한 번씩 존재하고 Fold별 개수가 Sinyoung 결과와 같은지 검사합니다. 여기서 통과해야 이후 Validation 점수를 팀원과 비교할 수 있습니다.

In [ ]:
import ast
import hashlib
import json
import math
import random
import re
import shutil
from itertools import permutations

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
VALIDATION_FOLD = 0
EXPECTED_FOLD_COUNTS = {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}
EXPECTED_TRAIN_COUNT = 7620
EXPECTED_VALIDATION_COUNT = 1915

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

train = pd.read_csv(TRAIN_CSV, dtype=str, encoding='utf-8-sig')
folds = pd.read_csv(FOLD_CSV, encoding='utf-8-sig')
assert list(folds.columns) == ['Id', 'fold'], folds.columns.tolist()
assert len(train) == len(folds) == 9535
assert train['Id'].is_unique and folds['Id'].is_unique
assert folds['Id'].notna().all() and folds['fold'].notna().all()
folds['fold'] = folds['fold'].astype(int)
assert folds['fold'].value_counts().sort_index().to_dict() == EXPECTED_FOLD_COUNTS

train = train.merge(folds, on='Id', how='left', validate='one_to_one')
assert train['fold'].notna().all()
train_fold0 = train[train['fold'] != VALIDATION_FOLD].reset_index(drop=True)
validation_fold0 = train[train['fold'] == VALIDATION_FOLD].reset_index(drop=True)
assert len(train_fold0) == EXPECTED_TRAIN_COUNT
assert len(validation_fold0) == EXPECTED_VALIDATION_COUNT
assert set(train_fold0['Id']).isdisjoint(set(validation_fold0['Id']))

def parse_answer(value):
    answer = ast.literal_eval(value) if isinstance(value, str) else list(value)
    assert isinstance(answer, list) and len(answer) == 4
    assert sorted(answer) == [1, 2, 3, 4]
    return [int(number) for number in answer]

assert train['Answer'].apply(parse_answer).map(tuple).nunique() == 24
fold_sha256 = hashlib.sha256(FOLD_CSV.read_bytes()).hexdigest()
print(f'Fold SHA-256: {fold_sha256}')
print(f'Fold 0 Train: {len(train_fold0):,}')
print(f'Fold 0 Validation: {len(validation_fold0):,}')
print('공통 Fold 감사 통과')

### 4. Train 전용 시간순서 증강 생성

각 Train ID에 원본 한 건과 결정적으로 선택한 비동일 순열 한 건을 만듭니다. 이미지 파일을 복사하지 않고 Input 파일명 순서만 바꾸며, 공식 `Answer`도 같은 순열로 다시 계산합니다. Validation은 원본 순서 1,915개만 유지합니다.

이 단계가 평가하는 것은 증강 후에도 네 이미지와 정답의 대응 관계가 정확히 유지되는지 여부입니다.

In [ ]:
IDENTITY_ORDER = (1, 2, 3, 4)
NON_IDENTITY_ORDERS = tuple(
    order for order in permutations(IDENTITY_ORDER) if order != IDENTITY_ORDER
)

def deterministic_order(sample_id, seed=SEED):
    digest = hashlib.sha256(f'{seed}:{sample_id}'.encode('utf-8')).digest()
    index = int.from_bytes(digest[:8], 'big') % len(NON_IDENTITY_ORDERS)
    return NON_IDENTITY_ORDERS[index]

def augment_row(row):
    result = row.copy()
    order = deterministic_order(row['Id'])
    old_inputs = [row[f'Input_{index}'] for index in IDENTITY_ORDER]
    old_answer = parse_answer(row['Answer'])
    for new_index, old_index in enumerate(order, start=1):
        result[f'Input_{new_index}'] = old_inputs[old_index - 1]
    new_answer = [old_answer[old_index - 1] for old_index in order]
    assert sorted(new_answer) == [1, 2, 3, 4]
    result['Answer'] = str(new_answer)
    result['_variant'] = 'augmented'
    result['_input_order'] = str(list(order))
    return result

original_train = train_fold0.copy()
original_train['_variant'] = 'original'
original_train['_input_order'] = str(list(IDENTITY_ORDER))
augmented_train = pd.DataFrame(
    [augment_row(row) for _, row in train_fold0.iterrows()]
)
train_examples = pd.concat([original_train, augmented_train], ignore_index=True)
validation_examples = validation_fold0.copy()
validation_examples['_variant'] = 'original'
validation_examples['_input_order'] = str(list(IDENTITY_ORDER))

assert len(train_examples) == 15240
assert train_examples['_variant'].value_counts().to_dict() == {
    'original': 7620, 'augmented': 7620
}
assert len(validation_examples) == 1915
assert (validation_examples['_variant'] == 'original').all()
assert train_examples['Answer'].apply(parse_answer).map(tuple).nunique() == 24

audit_index = 0
print('원본 Input:', [original_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('원본 Answer:', original_train.iloc[audit_index]['Answer'])
print('증강 Input 순서:', augmented_train.iloc[audit_index]['_input_order'])
print('증강 Input:', [augmented_train.iloc[audit_index][f'Input_{i}'] for i in IDENTITY_ORDER])
print('증강 Answer:', augmented_train.iloc[audit_index]['Answer'])
print(f'Train examples: {len(train_examples):,}')
print(f'Validation examples: {len(validation_examples):,}')

### 5. 조건부 프롬프트와 chronological target 정의

Caption의 시간 접속사 수에 따라 단계가 뚜렷한 동작과 연속 동작에 서로 다른 관찰 힌트를 줍니다. 모델 출력은 공식 Answer가 아니라 가장 이른 이미지부터 나열한 `chronological order`입니다. 학습과 추론에서 같은 의미를 사용하고, 평가 직전에 공식 Answer 형식으로 역변환합니다.

In [ ]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    'This caption describes distinct sequential stages, often marked by words like '
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def answer_to_chronological_order(answer):
    answer = parse_answer(answer)
    return [index for _, index in sorted((position, index) for index, position in enumerate(answer, 1))]

def chronological_order_to_answer(order):
    assert isinstance(order, list) and sorted(order) == [1, 2, 3, 4]
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

def build_prompt_messages(row, image_dir):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = image_dir / row['Id'] / row[f'Input_{image_number}']
        assert image_path.is_file(), image_path
        content.append({'type': 'image', 'image': str(image_path)})
        content.append({'type': 'text', 'text': f'\nImage {image_number}\n'})
    content.append({
        'type': 'text',
        'text': (
            f'Thinking about the sentence: \"{sentence}\"\n'
            f'{hint}\n'
            'Look at the 4 images above labeled Image 1 to Image 4. '
            'Determine the correct chronological order of these images to match the sentence. '
            'Provide the answer ONLY as a Python list of integers. '
            'Example: [1, 2, 3, 4]'
        ),
    })
    return [{'role': 'user', 'content': content}]

def build_training_messages(row, image_dir):
    messages = build_prompt_messages(row, image_dir)
    target = answer_to_chronological_order(row['Answer'])
    messages.append({
        'role': 'assistant',
        'content': [{'type': 'text', 'text': str(target)}],
    })
    return messages

for answer in permutations(IDENTITY_ORDER):
    answer = list(answer)
    assert chronological_order_to_answer(answer_to_chronological_order(answer)) == answer
print('24개 Answer ↔ chronological order 왕복 변환 통과')

### 6. Dataset과 completion-only loss 정의

모델에는 네 이미지, Caption, 프롬프트가 모두 입력되지만 loss는 assistant가 출력해야 하는 네 숫자 토큰에만 적용합니다. 프롬프트 토큰까지 정답으로 학습하는 구현 오류를 막기 위해 첫 배치에서 마스킹을 직접 감사합니다.

In [ ]:
from dataclasses import dataclass
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class OrderingDataset(Dataset):
    def __init__(self, dataframe, image_dir, training):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.training = training

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        messages = (
            build_training_messages(row, self.image_dir)
            if self.training
            else build_prompt_messages(row, self.image_dir)
        )
        return {'Id': row['Id'], 'messages': messages, 'Answer': row['Answer']}

@dataclass
class CompletionOnlyCollator:
    processor: object

    def __call__(self, examples):
        assert len(examples) == 1, '학습 batch size는 1로 고정합니다.'
        messages = examples[0]['messages']
        assert len(messages) == 2
        full_text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        prompt_text = self.processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        assert image_inputs is not None and len(image_inputs) == 4
        full = self.processor(
            text=[full_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors='pt',
        )
        prompt = self.processor(
            text=[prompt_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors='pt',
        )
        prompt_length = int(prompt['attention_mask'][0].sum().item())
        full_length = int(full['attention_mask'][0].sum().item())
        assert prompt_length < full_length
        assert torch.equal(
            full['input_ids'][0, :prompt_length],
            prompt['input_ids'][0, :prompt_length],
        ), 'Prompt가 전체 학습 문장의 prefix가 아닙니다.'
        labels = full['input_ids'].clone()
        labels[full['attention_mask'] == 0] = -100
        labels[:, :prompt_length] = -100
        assert int((labels != -100).sum().item()) > 0
        full['labels'] = labels
        return dict(full)

train_dataset = OrderingDataset(train_examples, TRAIN_IMAGE_DIR, training=True)
validation_dataset = OrderingDataset(validation_examples, TRAIN_IMAGE_DIR, training=False)
print(f'Train Dataset: {len(train_dataset):,}')
print(f'Validation Dataset: {len(validation_dataset):,}')

### 7. Qwen3-VL-8B와 7개 LoRA 모듈 로드

기본 모델 전체를 업데이트하지 않고 attention의 `q/k/v/o_proj`와 MLP의 `gate/up/down_proj`에만 LoRA를 연결합니다. `r=16`, `alpha=32`, `dropout=0.05`를 사용합니다. 모델 구조에서 7개 모듈이 실제로 모두 발견되는지도 검사합니다.

In [ ]:
import transformers
import peft
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, PeftModel, TaskType, get_peft_model

LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'right'

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
available_suffixes = {name.rsplit('.', 1)[-1] for name, _ in base_model.named_modules()}
missing_targets = sorted(set(LORA_TARGET_MODULES) - available_suffixes)
assert not missing_targets, f'모델에서 LoRA 대상 모듈을 찾지 못했습니다: {missing_targets}'

if FINAL_ADAPTER_DIR.is_dir():
    model = PeftModel.from_pretrained(base_model, FINAL_ADAPTER_DIR, is_trainable=False)
    TRAINING_REQUIRED = False
    print('저장된 final_adapter를 불러왔습니다. 학습 셀은 건너뜁니다.')
else:
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias='none',
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(base_model, lora_config)
    TRAINING_REQUIRED = True

model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()
model.print_trainable_parameters()
print('LoRA 대상:', LORA_TARGET_MODULES)

### 8. 첫 배치 loss 마스킹 감사

실제 모델 입력 한 건을 만들고 assistant 정답 토큰만 label로 남았는지 확인합니다. 이 셀은 forward/backward 학습을 수행하지 않으므로 빠르게 끝나야 합니다.

In [ ]:
collator = CompletionOnlyCollator(processor)
audit_batch = collator([train_dataset[0]])
supervised_tokens = int((audit_batch['labels'] != -100).sum().item())
non_padding_tokens = int(audit_batch['attention_mask'].sum().item())
assert 0 < supervised_tokens < non_padding_tokens
print(f'Assistant supervised tokens: {supervised_tokens}')
print(f'전체 non-padding tokens: {non_padding_tokens}')
print('Completion-only loss 감사 통과')
del audit_batch
torch.cuda.empty_cache()

### 9. Fold 0 LoRA 학습

원본 7,620개와 shuffled copy 7,620개를 한 epoch 학습합니다. batch size 1과 gradient accumulation 8을 사용하므로 optimizer가 보는 effective batch size는 8입니다. 250 optimizer step마다 체크포인트를 Drive에 저장하며, 셀을 다시 실행하면 마지막 체크포인트에서 재개합니다.

이 셀의 loss는 정답 문자열 생성 loss입니다. 대회 Exact Match는 다음 Validation 생성 셀에서 별도로 측정합니다.

In [ ]:
EPOCHS = 1.0
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
WARMUP_RATIO = 0.03
SAVE_STEPS = 250

run_config = {
    'scope': 'train_fold0_only_no_test_access',
    'fold_sha256': fold_sha256,
    'validation_fold': VALIDATION_FOLD,
    'unique_train_samples': len(train_fold0),
    'training_examples': len(train_examples),
    'validation_samples': len(validation_examples),
    'augmentation': 'original_plus_one_deterministic_non_identity_permutation',
    'model_name': MODEL_NAME,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'effective_batch_size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'lora': {
        'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
        'target_modules': LORA_TARGET_MODULES,
    },
}
with (RUN_DIR / 'run_config.json').open('w', encoding='utf-8') as file:
    json.dump(run_config, file, ensure_ascii=False, indent=2)

if TRAINING_REQUIRED:
    training_args = TrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type='cosine',
        optim='adamw_bnb_8bit',
        bf16=True,
        tf32=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        eval_strategy='no',
        save_strategy='steps',
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        logging_strategy='steps',
        logging_steps=10,
        logging_first_step=True,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        report_to='none',
        seed=SEED,
        data_seed=SEED,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator,
    )
    last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR)) if CHECKPOINT_DIR.is_dir() else None
    print(f'재개 체크포인트: {last_checkpoint or "없음 — 새 학습"}')
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(FINAL_ADAPTER_DIR))
    processor.save_pretrained(FINAL_ADAPTER_DIR)
    trainer.save_state()
    training_summary = {
        'train_metrics': train_result.metrics,
        'final_adapter_dir': str(FINAL_ADAPTER_DIR),
        'exact_match_evaluated': False,
    }
    with (RUN_DIR / 'training_summary.json').open('w', encoding='utf-8') as file:
        json.dump(training_summary, file, ensure_ascii=False, indent=2)
    print(f'학습 완료: {FINAL_ADAPTER_DIR}')
else:
    print(f'기존 학습 완료 adapter 사용: {FINAL_ADAPTER_DIR}')

### 10. 전체 Fold 0 Validation 생성 평가

학습에 사용하지 않은 Fold 0의 1,915개 원본 샘플에 greedy generation을 수행합니다. 가장 중요한 지표는 공식 Answer 네 자리가 모두 맞은 비율인 `Exact Match Accuracy`입니다. Position Accuracy, Pairwise Accuracy, Kendall tau는 오류가 얼마나 가까운지 진단하는 보조 지표입니다.

In [ ]:
def parse_chronological_output(text):
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or start >= end:
        return [1, 2, 3, 4], False, 'missing_brackets'
    try:
        value = ast.literal_eval(text[start:end + 1])
    except (SyntaxError, ValueError):
        return [1, 2, 3, 4], False, 'invalid_list'
    if not isinstance(value, list) or len(value) != 4 or sorted(value) != [1, 2, 3, 4]:
        return [1, 2, 3, 4], False, 'not_permutation'
    return chronological_order_to_answer([int(number) for number in value]), True, None

def compute_ordering_metrics(targets, predictions):
    targets = np.asarray(targets, dtype=int)
    predictions = np.asarray(predictions, dtype=int)
    exact = float(np.mean(np.all(targets == predictions, axis=1)))
    position = float(np.mean(targets == predictions))
    pairwise_values = []
    for target, prediction in zip(targets, predictions):
        correct = 0
        total = 0
        for first in range(4):
            for second in range(first + 1, 4):
                correct += (prediction[first] < prediction[second]) == (target[first] < target[second])
                total += 1
        pairwise_values.append(correct / total)
    pairwise = float(np.mean(pairwise_values))
    return {
        'exact_match_accuracy': exact,
        'position_accuracy': position,
        'pairwise_ordering_accuracy': pairwise,
        'mean_kendall_tau': 2.0 * pairwise - 1.0,
    }

model.config.use_cache = True
model.eval()
processor.tokenizer.padding_side = 'left'
EVAL_BATCH_SIZE = 4
results = []

for start_index in tqdm(range(0, len(validation_examples), EVAL_BATCH_SIZE), desc='Fold 0 Validation'):
    batch_rows = validation_examples.iloc[start_index:start_index + EVAL_BATCH_SIZE]
    batch_messages = [
        build_prompt_messages(row, TRAIN_IMAGE_DIR)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in batch_messages
    ]
    all_images = []
    for messages in batch_messages:
        image_inputs, video_inputs = process_vision_info(messages)
        assert not video_inputs
        all_images.extend(image_inputs)
    inputs = processor(
        text=batch_texts, images=all_images, padding=True, return_tensors='pt'
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs, max_new_tokens=16, do_sample=False, num_beams=1
        )
    generated_only = generated[:, inputs['input_ids'].shape[1]:]
    output_texts = processor.batch_decode(generated_only, skip_special_tokens=True)
    for (_, row), raw_output in zip(batch_rows.iterrows(), output_texts):
        prediction, parse_ok, parse_error = parse_chronological_output(raw_output)
        results.append({
            'Id': row['Id'],
            'target_answer': str(parse_answer(row['Answer'])),
            'predicted_answer': str(prediction),
            'parse_success': parse_ok,
            'parse_error': parse_error,
            'raw_output': raw_output,
        })

assert len(results) == EXPECTED_VALIDATION_COUNT
targets = [ast.literal_eval(row['target_answer']) for row in results]
predictions = [ast.literal_eval(row['predicted_answer']) for row in results]
metrics = compute_ordering_metrics(targets, predictions)
parse_success_rate = float(np.mean([row['parse_success'] for row in results]))
identity_mask = np.array([target == [1, 2, 3, 4] for target in targets])
shuffled_exact = float(np.mean(
    np.all(np.asarray(targets)[~identity_mask] == np.asarray(predictions)[~identity_mask], axis=1)
))

validation_predictions_path = RUN_DIR / 'fold0_validation_predictions.csv'
validation_summary_path = RUN_DIR / 'fold0_validation_summary.json'
pd.DataFrame(results).to_csv(validation_predictions_path, index=False, encoding='utf-8-sig')
validation_summary = {
    'scope': 'train_fold0_validation_only_no_test_access',
    'sample_count': len(results),
    'fold_sha256': fold_sha256,
    'metrics': metrics,
    'shuffled_only_exact_match_accuracy': shuffled_exact,
    'parse_success_rate': parse_success_rate,
    'adapter': str(FINAL_ADAPTER_DIR),
}
with validation_summary_path.open('w', encoding='utf-8') as file:
    json.dump(validation_summary, file, ensure_ascii=False, indent=2)

print(json.dumps(validation_summary, ensure_ascii=False, indent=2))
print(f'예측 저장: {validation_predictions_path}')
print(f'요약 저장: {validation_summary_path}')

### 11. 다음 판단 기준

우선 `parse_success_rate=1.0`인지 확인하고 Exact Match를 기존 결과와 비교합니다. 이 실험이 개선되면 augmentation 비율이나 epoch를 한 번에 하나씩 변경합니다. Validation 결과를 확인하기 전에는 Test 추론이나 제출 파일 생성을 진행하지 않습니다.